# SingBERT Inference — NS Commitment (Stage 5b — Full Corpus, Dual-Axis, Two Separate Models)

Runs BOTH distilled single-head commitment models over all ~737k NS Reddit chunks in a single pass.
Each model was trained on its own optimised training pool:
- **Buyin model** (`singbert_ns_buyin/best_model`): committed / uncommitted / neutral
- **Stance model** (`singbert_ns_stance/best_model`): supportive / critical / neutral

## What this produces
- `chunk_commitment_llm.parquet` — per chunk:
  - `buyin_label`, `stance_label`
  - `prob_buyin_committed`, `prob_buyin_uncommitted`, `prob_buyin_neutral`
  - `prob_stance_supportive`, `prob_stance_critical`, `prob_stance_neutral`
  - `is_positive` (committed OR supportive), `is_negative` (uncommitted OR critical)

## Kaggle datasets required
- Chunks dataset: `comments_chunks.parquet` + `submissions_chunks.parquet`
- Buyin model dir: `singbert_ns_buyin/best_model/` — upload output from `kaggle_distill_buyin_v1.ipynb`
- Stance model dir: `singbert_ns_stance/best_model/` — upload output from `kaggle_distill_stance_v1.ipynb`

## Notes
- chunk_ids are deduped across the two parquets (~1,545 known duplicates)
- Each model's `id2label.json` is read and asserted before inference
- Runtime: ~4–6 hrs on Kaggle T4 x2 at batch_size=128 (two full-corpus passes)

In [ ]:
!pip install -q -U transformers

In [ ]:
import glob
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Kaggle input root (datasets stored under /kaggle/input/datasets/kevinnchan/) ──
_KAGGLE_USER_DIR = Path("/kaggle/input/datasets/kevinnchan")

def find_input_file(*filenames: str) -> Path:
    for filename in filenames:
        # Check user dataset dir first
        matches = list(_KAGGLE_USER_DIR.glob(f"**/{filename}"))
        if matches:
            return matches[0]
        # Fallback: standard /kaggle/input
        matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
        if matches:
            return Path(matches[0])
        local = Path(filename)
        if local.exists():
            return local
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input/.")

def find_input_dir(*dirnames: str) -> Path:
    for dirname in dirnames:
        matches = [p for p in _KAGGLE_USER_DIR.glob(f"**/{dirname}") if p.is_dir()]
        if matches:
            return matches[0]
        matches = [Path(m) for m in glob.glob(f"/kaggle/input/**/{dirname}", recursive=True)
                   if Path(m).is_dir()]
        if matches:
            return matches[0]
    raise FileNotFoundError(f"None of {dirnames} found under /kaggle/input/.")

OUTPUT_DIR = Path("/kaggle/working")

# ── Config ────────────────────────────────────────────────────────────────
MAX_LEN    = 256
BATCH_SIZE = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"N GPUs : {torch.cuda.device_count()}")

In [ ]:
# ── Load buyin model → GPU 0 ──────────────────────────────────────────────
BUYIN_MODEL_DIR = find_input_dir("singbert-ns-buyin-bestmodel")
if (BUYIN_MODEL_DIR / "best_model").is_dir():
    BUYIN_MODEL_DIR = BUYIN_MODEL_DIR / "best_model"
print(f"Loading buyin model from: {BUYIN_MODEL_DIR}")

with open(BUYIN_MODEL_DIR / "id2label.json") as f:
    buyin_meta = json.load(f)

assert buyin_meta.get("axis") == "buyin", (
    f"Expected axis='buyin', got '{buyin_meta.get('axis')}'. Wrong model attached."
)
BUYIN_ID2LABEL = {int(k): v for k, v in buyin_meta["id2label"].items()}
BUYIN_LABEL2ID = {v: int(k) for k, v in BUYIN_ID2LABEL.items()}
assert set(BUYIN_ID2LABEL.values()) == {"committed", "uncommitted", "neutral"}

BUYIN_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
buyin_tokenizer = AutoTokenizer.from_pretrained(str(BUYIN_MODEL_DIR))
buyin_model = AutoModelForSequenceClassification.from_pretrained(str(BUYIN_MODEL_DIR))
buyin_model = buyin_model.to(BUYIN_DEVICE)
buyin_model.eval()
print(f"Buyin model loaded on {BUYIN_DEVICE}. Labels: {BUYIN_ID2LABEL}")

In [ ]:
# ── Load stance model → GPU 1 ──────────────────────────────────────────────
STANCE_MODEL_DIR = find_input_dir("singbert-ns-stance-bestmodel")
if (STANCE_MODEL_DIR / "best_model").is_dir():
    STANCE_MODEL_DIR = STANCE_MODEL_DIR / "best_model"
print(f"Loading stance model from: {STANCE_MODEL_DIR}")

with open(STANCE_MODEL_DIR / "id2label.json") as f:
    stance_meta = json.load(f)

assert stance_meta.get("axis") == "stance", (
    f"Expected axis='stance', got '{stance_meta.get('axis')}'. Wrong model attached."
)
STANCE_ID2LABEL = {int(k): v for k, v in stance_meta["id2label"].items()}
STANCE_LABEL2ID = {v: int(k) for k, v in STANCE_ID2LABEL.items()}
assert set(STANCE_ID2LABEL.values()) == {"supportive", "critical", "neutral"}

# Use cuda:1 if available, otherwise share cuda:0 with buyin
STANCE_DEVICE = torch.device("cuda:1" if torch.cuda.device_count() > 1 else
                              "cuda:0" if torch.cuda.is_available() else "cpu")
stance_tokenizer = AutoTokenizer.from_pretrained(str(STANCE_MODEL_DIR))
stance_model = AutoModelForSequenceClassification.from_pretrained(str(STANCE_MODEL_DIR))
stance_model = stance_model.to(STANCE_DEVICE)
stance_model.eval()
print(f"Stance model loaded on {STANCE_DEVICE}. Labels: {STANCE_ID2LABEL}")

In [ ]:
# ── Load all chunks (dedupe — ~1,545 known dupes at chunker batch boundaries) ─
comments_path    = find_input_file("comments_chunks.parquet")
submissions_path = find_input_file("submissions_chunks.parquet")

comments    = pd.read_parquet(comments_path,    columns=["chunk_id", "text"])
submissions = pd.read_parquet(submissions_path, columns=["chunk_id", "text"])

chunks = pd.concat([comments, submissions], ignore_index=True)
n_before = len(chunks)
chunks = chunks.drop_duplicates(subset="chunk_id").reset_index(drop=True)
chunks["text"] = chunks["text"].fillna("").astype(str)

print(f"Total rows loaded   : {n_before:,}")
print(f"  comments          : {len(comments):,}")
print(f"  submissions       : {len(submissions):,}")
print(f"Duplicates removed  : {n_before - len(chunks):,}  (expected ~1,545)")
print(f"Unique chunks       : {len(chunks):,}  (expected ~737k)")
assert chunks["chunk_id"].nunique() == len(chunks), "Dedupe failed"

In [ ]:
# ── Inference helper ──────────────────────────────────────────────────────
class ChunkDataset(Dataset):
    """Lazy tokenization — one sample at a time, avoids ~4.5 GB upfront RAM allocation."""
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {k: v.squeeze(0) for k, v in enc.items()}


def run_inference(texts, model, tokenizer, batch_size, device, label=""):
    """Returns softmax probability matrix [N, num_classes]."""
    dataset = ChunkDataset(texts, tokenizer, MAX_LEN)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2, pin_memory=True)

    all_probs = []
    total = len(loader)

    with torch.no_grad():
        for i, batch in enumerate(loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs["logits"]
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

            if (i + 1) % 100 == 0 or (i + 1) == total:
                done = min((i + 1) * batch_size, len(texts))
                print(f"  [{label}] {done:>8,} / {len(texts):,}  ({(i+1)/total*100:.1f}%)",
                      flush=True)

    return np.vstack(all_probs)

In [ ]:
# ── Run buyin + stance inference in parallel (one model per GPU) ──────────
import threading

texts = chunks["text"].tolist()
results = {}

def run_buyin():
    results["buyin"] = run_inference(texts, buyin_model, buyin_tokenizer,
                                     BATCH_SIZE, BUYIN_DEVICE, label="buyin")

def run_stance():
    results["stance"] = run_inference(texts, stance_model, stance_tokenizer,
                                      BATCH_SIZE, STANCE_DEVICE, label="stance")

print(f"Running BUYIN (GPU 0) + STANCE (GPU 1) in parallel on {len(chunks):,} chunks ...")
t_buyin  = threading.Thread(target=run_buyin)
t_stance = threading.Thread(target=run_stance)
t_buyin.start()
t_stance.start()
t_buyin.join()
t_stance.join()

buyin_probs  = results["buyin"]
stance_probs = results["stance"]

assert buyin_probs.shape  == (len(chunks), 3), "Buyin shape mismatch"
assert stance_probs.shape == (len(chunks), 3), "Stance shape mismatch"
print(f"Both done. buyin={buyin_probs.shape}  stance={stance_probs.shape}")

In [ ]:
# ── Post-processing: threshold + dual lexicon decode ──────────────────────
# Validated on 257-row human test set (2026-06-16):
#   uncommitted recall  0.419 → 0.767  (+0.348)
#   committed   recall  0.538 → 0.769  (+0.231)
#   kappa               0.524 → 0.713  (+0.189)
#   neutral precision   0.841 → 0.930  (+0.089)

BUYIN_THRESH_UNC = 0.06   # uncommitted fires if prob >= this AND is top class OR above threshold
BUYIN_THRESH_COM = 0.045  # committed fires if prob >= this AND is top class

# High-precision uncommitted signals — zero FP on test set
LEXICON_UNC = [
    "need to ooc", "oc and get",           # personal OOC intent
    "ord asap", "zao liao", "siam duty",   # Singlish avoidance
    "mc whenever", "took mc", "take mc",   # MC-seeking behaviour
    "avoid military", "avoid ns",
    "national slavery", "slavery", "slave",
    "against their will", "against his will",
    "concentration camp",
    "ns is a joke",
    "whats the point",
    "dont have a choice",
]

# Strong committed signals — can override even an uncommitted prediction
LEXICON_COM_STRONG = [
    "i signed on", "planning to sign on",  # voluntary enlistment / extension
    "officer scheme",                       # aspiring to officer track
    "up pes", "wanted to up",              # seeking higher fitness/active duty
    "pride after i ord", "belonging and pride",
    "reservist commitment",
]

# Soft committed signals — only flip neutral → committed
LEXICON_COM_SOFT = [
    "give your best",
]


def decode_buyin(probs_arr, id2label, texts,
                 thresh_unc=BUYIN_THRESH_UNC, thresh_com=BUYIN_THRESH_COM,
                 min_prob=0.0001):
    """
    Threshold + lexicon decode for buyin axis.
    Priority order:
      1. Strong committed lexicon  → committed  (overrides uncommitted too)
      2. Committed wins if it is the top class AND >= thresh_com
      3. Uncommitted threshold     → uncommitted (only when top class or unc >= thresh_unc)
      4. Committed threshold       → committed
      5. Soft committed lexicon    → committed  (neutral only)
      6. Uncommitted lexicon       → uncommitted (neutral only)
      7. argmax fallback           → neutral
    """
    label2id = {v: k for k, v in id2label.items()}
    i_unc = label2id["uncommitted"]
    i_com = label2id["committed"]

    labels = []
    for row_probs, text in zip(probs_arr, texts):
        text_l = text.lower()
        p_unc = row_probs[i_unc]
        p_com = row_probs[i_com]

        # Step 1: strong committed keyword overrides everything
        if any(kw in text_l for kw in LEXICON_COM_STRONG):
            labels.append("committed")
            continue

        # Step 2–4: threshold logic (committed protected if top class)
        if p_com >= p_unc and p_com >= thresh_com:
            label = "committed"
        elif p_unc >= thresh_unc:
            label = "uncommitted"
        elif p_com >= thresh_com:
            label = "committed"
        else:
            label = "neutral"

        # Step 5: soft committed lexicon (neutral only)
        if label == "neutral" and p_com >= min_prob:
            if any(kw in text_l for kw in LEXICON_COM_SOFT):
                label = "committed"

        # Step 6: uncommitted lexicon (neutral only)
        if label == "neutral" and p_unc >= min_prob:
            if any(kw in text_l for kw in LEXICON_UNC):
                label = "uncommitted"

        labels.append(label)

    return np.array(labels)


# Decode buyin with threshold + lexicon
buyin_labels_arr = decode_buyin(buyin_probs, BUYIN_ID2LABEL, chunks["text"].tolist())

# Stance: argmax (gate passes without tuning; kappa=0.596)
stance_pred_ids   = stance_probs.argmax(axis=1)
stance_labels_arr = np.array([STANCE_ID2LABEL[i] for i in stance_pred_ids])

# Combined flags
is_positive = ((buyin_labels_arr == "committed")   | (stance_labels_arr == "supportive"))
is_negative = ((buyin_labels_arr == "uncommitted") | (stance_labels_arr == "critical"))

out = pd.DataFrame({
    "chunk_id"                : chunks["chunk_id"],
    "buyin_label"             : buyin_labels_arr,
    "stance_label"            : stance_labels_arr,
    "prob_buyin_committed"    : buyin_probs[:, BUYIN_LABEL2ID["committed"]].astype("float32"),
    "prob_buyin_uncommitted"  : buyin_probs[:, BUYIN_LABEL2ID["uncommitted"]].astype("float32"),
    "prob_buyin_neutral"      : buyin_probs[:, BUYIN_LABEL2ID["neutral"]].astype("float32"),
    "prob_stance_supportive"  : stance_probs[:, STANCE_LABEL2ID["supportive"]].astype("float32"),
    "prob_stance_critical"    : stance_probs[:, STANCE_LABEL2ID["critical"]].astype("float32"),
    "prob_stance_neutral"     : stance_probs[:, STANCE_LABEL2ID["neutral"]].astype("float32"),
    "is_positive"             : is_positive,
    "is_negative"             : is_negative,
})

# Sanity checks
assert len(out) == len(chunks), "Row count mismatch"
assert out["chunk_id"].nunique() == len(out), "Duplicate chunk_ids in output"
buyin_sums  = out[["prob_buyin_committed","prob_buyin_uncommitted","prob_buyin_neutral"]].sum(axis=1)
stance_sums = out[["prob_stance_supportive","prob_stance_critical","prob_stance_neutral"]].sum(axis=1)
assert buyin_sums.between(0.999, 1.001).all(),  "Buyin softmax doesn't sum to 1"
assert stance_sums.between(0.999, 1.001).all(), "Stance softmax doesn't sum to 1"

out_path = OUTPUT_DIR / "chunk_commitment_llm.parquet"
out.to_parquet(out_path, index=False)
print(f"Saved -> {out_path}")
print(f"Rows   : {len(out):,}")
print(f"Columns: {list(out.columns)}")
print(f"\nDecode: buyin=threshold({BUYIN_THRESH_UNC}/{BUYIN_THRESH_COM})+lexicon, stance=argmax")

In [ ]:
# ── Distribution analysis ─────────────────────────────────────────────────
print(f"Joint distribution — buyin × stance:")
xt = pd.crosstab(out["buyin_label"], out["stance_label"], margins=True)
print(xt)
print()
pct = pd.crosstab(out["buyin_label"], out["stance_label"], normalize="all") * 100
print("Percentages:")
print(pct.round(2))

# ── Combined metric buckets ──────────────────────────────────────────────
print(f"\nCombined metric counts:")
n = len(out)
positive_count = is_positive.sum()
negative_count = is_negative.sum()
print(f"  is_positive (committed OR supportive): {positive_count:>8,}  ({positive_count/n*100:.1f}%)")
print(f"  is_negative (uncommitted OR critical): {negative_count:>8,}  ({negative_count/n*100:.1f}%)")

# ── 4 off-diagonal cells (the interesting combinations) ──────────────────
buyin_arr  = out["buyin_label"].values
stance_arr = out["stance_label"].values

buckets = {
    "committed   + supportive" : ((buyin_arr == "committed")   & (stance_arr == "supportive")).sum(),
    "committed   + critical"   : ((buyin_arr == "committed")   & (stance_arr == "critical")).sum(),
    "uncommitted + supportive" : ((buyin_arr == "uncommitted") & (stance_arr == "supportive")).sum(),
    "uncommitted + critical"   : ((buyin_arr == "uncommitted") & (stance_arr == "critical")).sum(),
}

print(f"\n4 off-diagonal cells:")
for label, count in buckets.items():
    print(f"  {label:<30}: {count:>8,}  ({count/n*100:.2f}%)")

print(f"\nInteresting combinations:")
print(f"  Frustrated loyalist   (committed + critical):     {buckets['committed   + critical']:,}")
print(f"  Apathetic supporter   (uncommitted + supportive): {buckets['uncommitted + supportive']:,}")

print(f"\nMean softmax probabilities:")
for c in ["prob_buyin_committed", "prob_buyin_uncommitted", "prob_buyin_neutral",
          "prob_stance_supportive", "prob_stance_critical", "prob_stance_neutral"]:
    print(f"  {c:<28}: {out[c].mean():.4f}")

print()
print("Next: download chunk_commitment_llm.parquet -> data/processed/new/, then")
print("  python -m scripts.rag.build_fact_table   (commitment axes now available)")
print("  and proceed to Stage 6 (document-level aggregation).")